# SITCOM-2014: Dome Speed, Acceleration, and Jerk analysis

We want to analyze the main dome’s Speed, Acceleration, and Jerk during the dome Endurance test. 
@Holger Drass is collecting the data and can send you a link to Google Drive. 
The requested Speed, Acceleration, and Jerk are in this ticket. 

{This test requires moving the dome at full speed (1.5 deg/s) for several movements.
We will have a test case linked to this ticket with more details once we have more information.
Since we will move fast, we need to confirm that the Dome Capacitors Banks are on and charged.
We will perform these movements via CSC, so we need someone who can drive the dome. 


* Associated ticket:
  [SITCOM-2014](https://rubinobs.atlassian.net/browse/SITCOM-2014)
  [SITCOM-1994](https://rubinobs.atlassian.net/browse/SITCOM-1994)

* Maximum Slew Rate:
    * Azimuth velocity: 2.25 deg/s
    * Azimuth acceleration: 1.125 deg/s^2
    * Azimuth jerk: 4.5 deg/s^3

* Minimum Slew Rate:
    * Azimuth velocity: 1.5 deg/s
    * Azimuth acceleration: 0.75 deg/s^2
    * Azimuth jerk: 3.0 deg/s^3

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from astropy.time import Time
from pathlib import Path
from datetime import datetime, timedelta
import csv
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import UnivariateSpline
from scipy.signal import find_peaks

from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

In [ ]:
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)
efd_client = makeEfdClient()

In [ ]:
unit = r'N$\cdot$m'

## Define the maximum and minimum speed of MTDome motion 

This is the requirement of the maximum and the minimum limit for each motion - velocity, acceleration, and jerk. 

In [ ]:
vel_max_limit = 2.25 #deg/s
accel_max_limit = 1.125 #deg/s^2
jerk_max_limit = 4.5 #deg/s^3
vel_min_limit = 1.5  #deg/s
accel_min_limit =  0.75  #deg/s^2
jerk_min_limit = 3.00   #deg/s^3

# Data Analysis
The timestamps of start and the end of the MTDome movement are below:<br>
Start <code>2025/03/27 10:37:38.987</code> <br>
End <code>2025/03/27 11:55:26.103</code>

However, it seems that the actual MTDome endurance test happened between <br>
Start <code>2025-03-27T10:40:00</code><br>
End <code>2025-03-27T11:40:00</code>

## Dome Endurance Test 


### Setting and Querying the Data for MTDome azimuth positoin and motion

In [ ]:
#DayObs and Time
dayobs="2025-03-27"
start_time= "2025-03-27T10:40:00"
end_time = "2025-03-27T11:40:00"

In [ ]:
#Changing the format to query
start_time= Time(start_time, scale="utc", format="isot")
end_time = Time(end_time, scale="utc", format="isot")

In [ ]:
#Querying the MTDome azimuth position and motion. Note that no data in velocityCommanded. 
dome_data = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTDome.azimuth",
    columns=["velocityActual","velocityCommanded", "positionActual", "positionCommanded","private_sndStamp"],
    begin=start_time,
    end= end_time,
    
)

### Query the Az Configuration Applied through EUI
* <code>lsst.sal.MTDome.logevent_azConfigurationApplied</code> only exists when there was any update applied. <br>
Iteratively querying it if there is no telemetry on the timespan. <br>
Iteratively change the <code>begin</code> time until the last applied value appears. <br>

* <code>lsst.sal.MTDome.logevent_azMotion </code> state for Dome Motion <br>
   <code>lsst.sal.MTDome.logevent_azMotion.state == 5 </code>: move <br>
   <code>lsst.sal.MTDome.logevent_azMotion.state == 4 </code>: crawl 


In [ ]:
#Querying the applied dome motion setup
dome_setup = pd.DataFrame()
while dome_setup.empty:
    dome_setup = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.logevent_azConfigurationApplied",
        columns=["amax","jmax","vmax"],
        begin = start_time-timedelta(hours=1),
        end= end_time,
        warn = False
)    

In [ ]:
#Querying the applied dome motion state. 
dome_state= getEfdData(
    client=efd_client,
    topic="lsst.sal.MTDome.logevent_azMotion",
    columns=["state","private_sndStamp"],
    begin=start_time,
    end= end_time,
)

In [ ]:
#Defining move and crawl state from enum 
dome_state_move = dome_state[dome_state.state == 5]
dome_state_crawl = dome_state[dome_state.state == 4]

## Calculating velocity, acceleration, and jerk

<code>lsst.sal.MTDome.azimuth</code> only has position and velocity.<br>
Acceleration (deg/s^2) and Jerk (deg/s^3) can be calculated from derivative of velocity. <br>
We can also get velocity (deg/s) from the derivative of position.<br>

In [ ]:
#copy the df to some calculation of position and motions
dome = dome_data.copy()

#Time converted to delta Time from the beginning of the test
time_seconds = (dome.private_sndStamp - dome.private_sndStamp.iloc[0]).values
position = dome.positionActual.values

#Convert to Rad and revert to degrees to remove the discontinuity in degrees. 
position_unwrapped = np.unwrap(np.deg2rad(position))
position_unwrapped_deg = np.rad2deg(position_unwrapped)

#An accumulated position
diff_position_unwrapped_deg = np.diff(position_unwrapped_deg) 
position_accumulate= np.cumsum(np.insert(diff_position_unwrapped_deg, 0, 0))
dome.insert(len(dome.columns)-1,'pos_acum', position_accumulate)

#Find the spline derivation to get velocity from the positions. 
spline = UnivariateSpline(time_seconds, position_unwrapped_deg, s=2)
velocity_spline = spline.derivative()(time_seconds)
dome.insert(len(dome.columns)-1,'vel_sp',velocity_spline)

#Find the spline derivation to get acceleration from the velocity
velocity = dome.velocityActual.values
spline = UnivariateSpline(time_seconds, velocity, s=2)
accel = spline.derivative()(time_seconds)
dome.insert(len(dome.columns)-1,'accel',accel)

#Find the spline derivation to get velocity from the positions. 
spline = UnivariateSpline(time_seconds, accel, s=2)
jerk = spline.derivative()(time_seconds)
dome.insert(len(dome.columns)-1,'jerk',jerk)

#Using a cumulative trapezoid to get the position from velocity.
position_cumtrap = cumulative_trapezoid(dome.velocityActual.values, dome.private_sndStamp.values-dome.private_sndStamp.values[0],initial=0)  # initial=0 sets position[0] = 0
position_cumtrap = position_cumtrap+dome.positionActual.values[0]
dome.insert(len(dome.columns)-1,'pos_cumtrap', position_cumtrap % 360) # When it exceeds 360, then back to 0. 

In [ ]:
#Setting the arbitrary number for the threshold of non-zero motion: abs(jerk) > 0.05 deg/s 
val = dome["private_sndStamp"].copy()
dome["delta_time_move"] = dome["move_flag"] = dome["velocity_integ"] = val
dome["move_flag"] = 0
dome["delta_time_move"] = 0.0
start_move = (np.abs(dome.jerk.shift(-1)) > 0.05) & (np.abs(dome.jerk) < 0.05)#  & (np.abs(dome.jerk) > np.abs(dome.jerk.shift(1))).shift(-1).convert_dtypes().fillna(False).astype(bool)
end_move = (np.abs(dome.jerk.shift(-1)) < 0.05) & (np.abs(dome.jerk) > 0.05) #& (np.abs(dome.jerk) < np.abs(dome.jerk.shift(1)))

## Actual Position and Position Calculated with Actual Velocity
This is to compare the <code>positionActual</code> and the position calculated from <code>velocityActual</code>. <br>
Note that there is a systematic offset between actual position and calculated position. <br>
This can be due to the way of integrating velocity with respect to time. 

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

x_plot_min = Time("2025-03-27T11:00:00", scale="utc", format="isot").datetime
x_plot_max = Time("2025-03-27T11:03:00", scale="utc", format="isot").datetime

ax.plot(dome.index, dome.positionActual,'-',label="Position (Actual)")
ax.plot(dome.index, dome.pos_cumtrap,':', label="Position (Calculated)",color="purple")
#ax.set_xlim([x_plot_min,x_plot_max])
#ax.set_ylim([240,270])

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Position (deg)")
ax.grid(which="major", axis="both", linestyle="--")

ax2 = ax.twinx()
xmin, xmax = ax2.get_xlim()
pos_diff = (dome.positionActual - dome.pos_cumtrap + 180) % 360 - 180
ax2.plot(dome.index,pos_diff,'.',
         label="Difference between Actual Velocity and Integrated", color="grey", markersize=1)
plt.title(f"Positions and Difference on {dayobs}")
ax2.set_ylabel("Actual - Calculated (deg)", rotation=270, labelpad=15)
lgd = fig.legend(bbox_to_anchor=(1.5, 0.3),fontsize=12)
ax2.text(0.85*xmin+0.15*xmax,0.4, r"$\mu_{diff}$"f": {np.mean(abs(pos_diff)):.2f} \u00B1 {np.std(abs(pos_diff)):.2f} ", horizontalalignment='center', verticalalignment='center', fontsize=12, color="Black")

fig.savefig(plot_path /f"{dayobs}_Position_Diff.png",bbox_extra_artists=(lgd,), bbox_inches='tight')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

x_plot_min = Time("2025-03-27T11:30:00", scale="utc", format="isot").datetime
x_plot_max = Time("2025-03-27T11:31:00", scale="utc", format="isot").datetime

ax.plot(dome.index, dome.positionActual,'.-',label="Position (Actual)")
ax.plot(dome.index, dome.pos_cumtrap,'.-', label="Position (Calculated)",color="purple")
ax.set_xlim([x_plot_min,x_plot_max])
ax.set_ylim([35,45])

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Position (deg)")
ax.grid(which="major", axis="both", linestyle="--")
xmin, xmax = ax.get_xlim() 
ymin, ymax = ax.get_ylim()
ax.text(0.85*xmin+0.15*xmax,0.85*ymax+0.15*ymin, r"$\mu_{diff}$"f": {np.mean(abs(pos_diff)):.2f} \u00B1 {np.std(abs(pos_diff)):.2f} ", horizontalalignment='center', verticalalignment='center', fontsize=12, color="Black")
ax2 = ax.twinx()
ax2.set_ylim([0.2,0.5])

pos_diff = (dome.positionActual - dome.pos_cumtrap + 180) % 360 - 180
ax2.plot(dome.index,pos_diff,'.',
         label="Difference between Actual Velocity and Integrated", color="black", markersize=1)
plt.title(f"Zoom in plot of Positions and Difference on {dayobs}")
ax2.set_ylabel("Actual - Calculated (deg)", rotation=270, labelpad=15)
lgd = fig.legend(bbox_to_anchor=(1.5, 0.3),fontsize=12)
plt.show()
fig.savefig(plot_path /f"{dayobs}_Position_Motion_change_zoom_in.png",bbox_extra_artists=(lgd,), bbox_inches='tight')

## How Position, Velocity, Acceleration, and Jerk change with respect to time 
This is a zoomed-in plot to see how position, velocity, acceleration, and jerk change. <br>
You can see that jerk and velocity do not reach the limit, but it seems acceleration reaches the maximum. 

In [ ]:
x_plot_min = Time("2025-03-27T11:00:29", scale="utc", format="isot").datetime
x_plot_max = Time("2025-03-27T11:00:37", scale="utc", format="isot").datetime

fig, ax = plt.subplots(figsize=(10,5))

ax.plot(dome.index, dome.positionActual,'.-', color="black",label="Position (Actual)")
#ax.plot(dome.index,dome.pos_cumtrap,'.:', color="black",label="Position (Calculated)")
ax.set_xlim([x_plot_min, x_plot_max])
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Position(deg)")
ax.grid(which="major", axis="both", linestyle="--")
ax.set_ylim([246,254])
ax2 = ax.twinx()
ymin, ymax = ax2.get_ylim()
xmin, xmax = ax2.get_xlim()
ax2.plot(dome.index, dome.velocityActual,'.-', color="purple", label="Velocity (Actual)")
#ax2.plot(dome.index, velocity_spline, '.:', color="grey", label="Velocity (Calculated)")

ax2.plot(dome.index, dome.accel,'.:', color='red', label="acceleration")
ax2.plot(dome.index, dome.jerk,'.:', color='green', label="jerk")
#ax2.plot(dome.index, dome.delta_time_move*dome_setup.amax.iloc[-1], '.',color='orange')


#velocity
ax2.hlines(dome_setup.vmax.iloc[-1],xmax=x_plot_max, xmin=x_plot_min, color='Blue')
ax2.text(0.87*xmin+0.13*xmax,dome_setup.vmax.iloc[-1]+0.25, 'Max Applied Velocity', 
         horizontalalignment='center', verticalalignment='center', fontsize=11, color="Blue")


#acceleration
ax2.hlines(dome_setup.amax.iloc[-1],xmax=x_plot_max, xmin=x_plot_min, color='red')
ax2.hlines(-1.0*dome_setup.amax.iloc[-1],xmax=x_plot_max, xmin=x_plot_min, color='red')
ax2.text(0.85*xmin+0.15*xmax,dome_setup.amax.iloc[-1]+0.25, "Max Applied Accelerlation", 
         horizontalalignment='center', verticalalignment='center', fontsize=11, color="red")
ax2.text(0.85*xmin+0.15*xmax,-1.0*dome_setup.amax.iloc[-1]-0.25, 'Max Applied Accelerlation', 
         horizontalalignment='center', verticalalignment='center', fontsize=11, color="red")

#jerk
ax2.hlines(dome_setup.jmax.iloc[-1],xmax=x_plot_max, xmin=x_plot_min, color='green')
ax2.hlines(-1.0*dome_setup.jmax.iloc[-1],xmax=x_plot_max, xmin=x_plot_min, color='green')

ax2.text(0.9*xmin+0.1*xmax,dome_setup.jmax.iloc[-1]+0.25, 'Max Applied Jerk', 
         horizontalalignment='center', verticalalignment='center', fontsize=11, color="green")
ax2.text(0.9*xmin+0.1*xmax,-1.0*dome_setup.jmax.iloc[-1]-0.25, 'Max Jerk Limit', 
         horizontalalignment='center', verticalalignment='center', fontsize=11, color="green")

# find peaks for acceleration and annotation
peaks_accel, properties_accel = find_peaks(abs(dome.accel), height=0.4, distance=2, prominence=0.1)

for peak in peaks_accel:
    if dome.accel.iloc[peak] < 0: 
        length = -0.7 
    else: 
        length = 0.7
    ax2.annotate(
        'deceleration', 
        xy=(dome.index[peak], dome.accel.iloc[peak]), 
        xytext=(dome.index[peak], dome.accel.iloc[peak] + length),
        arrowprops=dict(arrowstyle='->', color='black',linewidth=2),fontsize=11)
    #ax2.text(dome.index[peak], dome.accel.iloc[peak] + length*1.2, 'Deceleration', 
    #     horizontalalignment='center', verticalalignment='center', fontsize=11, color="black")    
    

ax2.set_ylim([-4,4])
#ax.vlines(dome_state_move.index, ymax=ymax,ymin=ymin, linestyle= ':', color="r",label="lsst.sal.MTDome.logevent_azMotion.State = Moving")
#ax.vlines(dome_state_crawl.index, ymax=ymax, ymin=ymin, color="g", label="lsst.sal.MTDome.logevent_azMotion.State = Crawling")

plt.title(f"Position, velocity, acceleration, and jerk\n dayobs: {dayobs}")
ax2.set_ylabel("Velocity(deg/s)/acceleration(deg/s2)/Jerk(deg/s3)", rotation=270, labelpad=15)
lgd = fig.legend(bbox_to_anchor=(1.2, 0.4),fontsize=12)


fig.savefig(plot_path /f"{dayobs}_Position_Motion_change.png",bbox_extra_artists=(lgd,), bbox_inches='tight')

plt.show()

## Position and Velocity
This plot presents position and velocity data, including the actual velocity and the velocity derived from a spline interpolation of the position

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(dome.index, dome.positionActual,',-', color="tab:green", label="Position (Acutal)")
        
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Position (deg)")
ax.grid(which="major", axis="both", linestyle="--")
ax.autoscale(enable=True, axis='x', tight=True)
ax2 = ax.twinx()
ax2.scatter(dome.index, dome.velocityActual, color='grey',s=1**3,label="velocityActual")
ax2.scatter(dome.index, dome.vel_sp,color='darkorange',s=1**3,label="Calculated Velocity")
ax2.plot(dome.index, dome.vel_sp-dome.velocityActual,',', color='lightcoral')

vel_diff = dome.vel_sp - dome.velocityActual

vel_diff_max_per_min = vel_diff.resample('1min').max()
vel_diff_min_per_min = vel_diff.resample('1min').min()


ax2.fill_between(vel_diff_max_per_min.index,vel_diff_max_per_min,vel_diff_min_per_min,
    color='lightcoral',
    alpha=0.3,
    label=r"${\Delta}$velocity"
)


ax2.set_ylabel("Velocity (deg/s)", rotation=270, labelpad=15)
lgd = fig.legend(bbox_to_anchor=(1.25, 0.4),fontsize=12)
plt.title(f"MTDome Endurance Test\n{dayobs}")
plt.show()
fig.savefig(plot_path /f"{dayobs}_MTDome_pos_vel.png",bbox_extra_artists=(lgd,), bbox_inches='tight')

## Velocity

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

#ax.plot(dome.index, dome.velocityActual,'.',color="black")
ax.scatter(dome.index, dome.velocityActual, c="grey", s = 1**3, label='Actual Velocity')
ax.scatter(dome.index, dome.vel_sp, c="darkorange", s = 1**3, label="Calculated Velocity")
#ax.plot(dome.index, dome.vel_cal,',:', color='red',label="Calculated Velocity")
ax.plot(dome.index, dome.vel_sp-dome.velocityActual,',', color="lightcoral")
ax.autoscale(enable=True, axis='x', tight=True)
xmin, xmax = ax.get_xlim()

vel_diff = dome.vel_sp - dome.velocityActual

vel_diff_max_per_min = vel_diff.resample('1min').max()
vel_diff_min_per_min = vel_diff.resample('1min').min()

ax.fill_between(vel_diff_max_per_min.index,vel_diff_max_per_min,vel_diff_min_per_min,
    color='lightcoral',
    alpha=0.3,
    label=r"${\Delta}$velocity"
)

peaks_vel, properties_vel = find_peaks(dome.velocityActual, height=1.25, distance=5, prominence=0.1)
peaks_mean = np.mean(dome.velocityActual.iloc[peaks_vel])
peaks_std = np.std(dome.velocityActual.iloc[peaks_vel])

ax.hlines(peaks_mean,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-', linewidth=0.75, label=r"${\mu}_{Peak\ Velocity}$")
ax.fill_betweenx([peaks_mean - peaks_std, peaks_mean + peaks_std], xmin, xmax, color='green', alpha=0.3,label=r"$1{\sigma}_{peak}$")


# Maximum Velocity Limit
ax.hlines(vel_max_limit,xmin=xmin, xmax=xmax,color='red', linestyle='-')
ax.text(0.85*xmin+0.15*xmax,vel_max_limit*0.95, 'Max Velocity Limit', horizontalalignment='center', verticalalignment='center', fontsize=12, color="red")

#Minimum Velocity Limit
ax.hlines(vel_min_limit,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-')
ax.text(0.85*xmin+0.15*xmax,vel_min_limit*1.05, 'Min Velocity Limit', horizontalalignment='center', verticalalignment='center', fontsize=12, color="darkgreen")

#Applied Velocity Limit
ax.hlines(dome_setup.vmax.iloc[-1],xmin=xmin, xmax=xmax,color='Blue', linestyle=':')
ax.text(0.25*xmin+0.75*xmax,dome_setup.vmax.iloc[-1]*1.05, 'Applied Max Velocity', horizontalalignment='center', verticalalignment='center', fontsize=12, color="Blue")

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Velocity (deg/s)")
ax.grid(which="major", axis="both", linestyle="--")
ax.autoscale(enable=True, axis='x', tight=True)
plt.title(f"MTDome Endurance Test - Velocity\n{dayobs}")
lgd = fig.legend(bbox_to_anchor=(1.2, 0.5),fontsize=12)
plt.show()
fig.savefig(plot_path /f"{dayobs}_MTDome_vel.png",bbox_extra_artists=(lgd,), bbox_inches='tight')
plt.show()

## The Time Difference Between Each Movement
This plot illustrates how position and velocity change to time. <br>
The state of the dome <code>move</code> and <code>crawl</code> were overlayed. <br>
Note that the transition of dome state does not help recognize or seperate the real dome Az motion. <br>

In [ ]:
x_plot_min = Time("2025-03-27T11:00:29", scale="utc", format="isot").datetime
x_plot_max = Time("2025-03-27T11:02:00", scale="utc", format="isot").datetime

fig, ax = plt.subplots(figsize=(10,5))

ax.plot(dome.index, dome.positionActual,'.-',label="Position")
ax.set_xlim([x_plot_min, x_plot_max])
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Position(deg)")
ax.grid(which="major", axis="both", linestyle="--")
ax.set_ylim([200,280])
ax2 = ax.twinx()
ax2.set_ylim([-1,2])
ymin, ymax = ax2.get_ylim()

ax2.plot(dome.index, dome.velocityActual,',-', color='black', label="Velocity")
ax2.vlines(dome_state_move.index ,ymin=ymin, ymax=ymax,color='red', linestyle=':',label="Dome State: Moving")
ax2.vlines(dome_state_crawl.index ,ymin=ymin, ymax=ymax,color='green', linestyle=':',label="Dome State: Crawling")


peaks_vel, properties_vel = find_peaks(dome.velocityActual, height=1.25, distance=5, prominence=0.1)
peak_loc = np.diff(dome.private_sndStamp.iloc[peaks_vel])

plt.title(f"The time between each peaks {dayobs}")
ax2.set_ylabel("Velocity(deg/s)", rotation=270, labelpad=15)
lgd = fig.legend(bbox_to_anchor=(1.2, 0.3),fontsize=12)
fig.savefig(plot_path /f"{dayobs}_delta_d_between_peaks.png",bbox_extra_artists=(lgd,), bbox_inches='tight')

plt.show()

In [ ]:
print(f"Is there any point exceeding the maximum velocity limit? {(properties_vel["peak_heights"] > dome_setup.vmax.iloc[-1]).any()}")
print(f"delta(t) between peaks:\n"
      f"mean: {np.mean(peak_loc):.2f}sec, stddev:{np.std(peak_loc):.2f}sec, "
      f"max:{np.min(peak_loc):.2f}sec, min:{np.max(peak_loc):.2f}sec")

## Acceleration
As there is no telemetry on acceleration, it was derived from a derivative of the <code>velocityActual</code>. 


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

ax.scatter(dome.index, dome.accel, c="grey", s = 1**3, label='Acceleration')
ax.set_ylim([-1.5,1.5])

ax.autoscale(enable=True, axis='x', tight=True)
xmin, xmax = ax.get_xlim()
# Maximum Acceleration Limit
ax.hlines(accel_max_limit,xmin=xmin, xmax=xmax,color='red', linestyle='-')
ax.hlines(-1*accel_max_limit,xmin=xmin, xmax=xmax,color='red', linestyle='-')

ax.text(0.85*xmin+0.15*xmax,accel_max_limit*-0.88, 'Max Acceleration Limit', horizontalalignment='center', verticalalignment='center',
        fontsize=12, color="red", bbox=dict(facecolor='white', alpha=0.6, edgecolor="None"))

#Minimum Acceleration Limit
ax.hlines(accel_min_limit,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-')
ax.hlines(-1*accel_min_limit,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-')

ax.text(0.85*xmin+0.15*xmax,accel_min_limit*-0.8, 'Min Acceleration Limit', horizontalalignment='center', verticalalignment='center', 
        fontsize=12, color="darkgreen", bbox=dict(facecolor='white', alpha=0.6, edgecolor="None"))

#Applied Acceleration Limit
ax.hlines(dome_setup.amax.iloc[-1],xmin=xmin, xmax=xmax,color='Blue', linestyle='-')
ax.hlines(-1*dome_setup.amax.iloc[-1],xmin=xmin, xmax=xmax,color='Blue', linestyle='-')
ax.text(0.15*xmin+0.85*xmax,dome_setup.amax.iloc[-1]*-1.2, 'Applied Max Acceleration', horizontalalignment='center', verticalalignment='center', 
        fontsize=12, bbox=dict(facecolor='white', alpha=0.6, edgecolor="None"),color="Blue")

peaks_accel_pos, peaks_accel_properties_pos = find_peaks(dome.accel, height=0.5, distance=5, prominence=0.1)
peaks_accel_neg, peaks_accel_properties_neg = find_peaks(-dome.accel, height=0.5, distance=5, prominence=0.1)
peaks_mean_pos = np.mean(dome.accel.iloc[peaks_accel_pos])
peaks_mean_neg = np.mean(dome.accel.iloc[peaks_accel_neg])
peaks_std_pos = np.std(dome.accel.iloc[peaks_accel_pos])
peaks_std_neg = np.std(dome.accel.iloc[peaks_accel_neg])


ax.hlines(peaks_mean_pos,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-', linewidth=0.75, label=r"${\mu}_{Peak\ Acceleration}$")
ax.hlines(peaks_mean_neg,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-', linewidth=0.75)
ax.fill_betweenx([peaks_mean_pos - peaks_std_pos, peaks_mean_pos + peaks_std_pos], xmin, xmax, color='green', alpha=0.3,label=r"$1{\sigma}_{peak}$")
ax.fill_betweenx([peaks_mean_neg - peaks_std_neg, peaks_mean_neg + peaks_std_neg], xmin, xmax, color='green', alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Acceleration (deg/s$^{2}$)")
ax.grid(which="major", axis="both", linestyle="--")
ax.autoscale(enable=True, axis='x', tight=True)
plt.title(f"MTDome Endurance Test - Acceleration\n{dayobs}")
lgd = fig.legend(bbox_to_anchor=(1.2, 0.5),fontsize=12)
plt.show()
fig.savefig(plot_path /f"{dayobs}_MTDome_accel.png",bbox_extra_artists=(lgd,), bbox_inches='tight')
plt.show()

In [ ]:
peaks_accel, properties_accel = find_peaks(abs(dome.accel), height=0.4, distance=2, prominence=0.1)
peak_accel_loc = np.diff(dome.private_sndStamp.iloc[peaks_accel])
print(f"Is there any point violent the maximum acceleration limit? {(properties_accel["peak_heights"] > accel_max_limit).any()}")
print(f"How many points? {np.sum(properties_accel["peak_heights"] > accel_max_limit)}")
print(f"Is there any point higher than applied max acceleration? {(properties_accel["peak_heights"] > dome_setup.amax.iloc[-1]).any()}")
print(f"How many points? {np.sum(properties_accel["peak_heights"] > dome_setup.amax.iloc[-1])}")

There is only one point where the acceleration exceeds the maximum acceleration limit. <br> 
However, it is hard to say as acceleration was derived from a derivative of velocity. <br>

## Jerk
As there is no telemetry of actual jerk, jerk here is the second derivative of <code>positionActual</code>. <br>
Jerk is lower than even the minimum requirement, but this could be because there was no long slew or motion in Azimuth. <br>

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

ax.scatter(dome.index, dome.jerk, c="grey", s = 1**3, label='Jerk')
ax.set_ylim([-6,6])

ax.autoscale(enable=True, axis='x', tight=True)
xmin, xmax = ax.get_xlim()

# Maximum Jerk Limit
ax.hlines(jerk_max_limit,xmin=xmin, xmax=xmax,color='red', linestyle='-')
ax.hlines(-1*jerk_max_limit,xmin=xmin, xmax=xmax,color='red', linestyle='-')

ax.text(0.85*xmin+0.15*xmax,jerk_max_limit*-0.88, 'Max Jerk Limit', horizontalalignment='center', verticalalignment='center',
        fontsize=12, color="red", bbox=dict(facecolor='white', alpha=0.6, edgecolor="None"))

#Minimum jerk Limit
ax.hlines(jerk_min_limit,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-')
ax.hlines(-1*jerk_min_limit,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-')

ax.text(0.85*xmin+0.15*xmax,jerk_min_limit*-0.8, 'Min Jerk Limit', horizontalalignment='center', verticalalignment='center', 
        fontsize=12, color="darkgreen", bbox=dict(facecolor='white', alpha=0.6, edgecolor="None"))

#Applied Velocity Limit
ax.hlines(dome_setup.jmax.iloc[-1],xmin=xmin, xmax=xmax,color='Blue', linestyle=':')
ax.hlines(-1*dome_setup.jmax.iloc[-1],xmin=xmin, xmax=xmax,color='Blue', linestyle=':')
ax.text(0.15*xmin+0.85*xmax,dome_setup.jmax.iloc[-1]*-1.2, 'Applied Max Jerk', horizontalalignment='center', verticalalignment='center', 
        fontsize=12, bbox=dict(facecolor='white', alpha=0.6, edgecolor="None"),color="Blue")


peaks_jerk_pos,_ = find_peaks(dome.jerk, height=0.1, distance=5, prominence=0.1)
peaks_jerk_neg,_ = find_peaks(-dome.jerk, height=0.1, distance=5, prominence=0.1)

peaks_mean_pos = np.mean(dome.jerk.iloc[peaks_jerk_pos])
peaks_mean_neg = np.mean(dome.jerk.iloc[peaks_jerk_neg])

peaks_std_pos = np.std(dome.jerk.iloc[peaks_jerk_pos])
peaks_std_neg = np.std(dome.jerk.iloc[peaks_jerk_neg])

ax.hlines(peaks_mean_pos,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-', linewidth=0.75, label=r"${\mu}_{Peak\ Jerk}$")
ax.hlines(peaks_mean_neg,xmin=xmin, xmax=xmax,color='darkgreen', linestyle='-', linewidth=0.75)
ax.fill_betweenx([peaks_mean_pos - peaks_std_pos, peaks_mean_pos + peaks_std_pos], xmin, xmax, color='green', alpha=0.3,label=r"$1{\sigma}_{peak}$")
ax.fill_betweenx([peaks_mean_neg - peaks_std_neg, peaks_mean_neg + peaks_std_neg], xmin, xmax, color='green', alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlabel("Time")
ax.set_ylabel("Jerk (deg/s$^{3}$)")
ax.grid(which="major", axis="both", linestyle="--")
ax.autoscale(enable=True, axis='x', tight=True)
plt.title(f"MTDome Endurance Test - Jerk \n{dayobs}")
fig.legend(bbox_to_anchor=(1.1, 0.3),fontsize=12)
#fig.legend(bbox_to_anchor=(0.82, 0.87),fontsize=12)
fig.savefig(plot_path /f"{dayobs}_MTDome_jerk.png",bbox_extra_artists=(lgd,), bbox_inches='tight')
plt.show()

In [ ]:
peaks_jerk, properties_jerk = find_peaks(abs(dome.jerk), height=0.4, distance=2, prominence=0.1)
peak_jerk_loc = np.diff(dome.private_sndStamp.iloc[peaks_jerk])
print(f"Is there any point violent the maximum acceleration limit? {(properties_jerk["peak_heights"] > jerk_max_limit).any()}")
print(f"How many points? {np.sum(properties_jerk["peak_heights"] > jerk_max_limit)}")
print(f"Is there any point higher than applied max jerk? {(properties_jerk["peak_heights"] > dome_setup.jmax.iloc[-1]).any()}")
print(f"How many points? {np.sum(properties_jerk["peak_heights"] > dome_setup.jmax.iloc[-1])}")

## Histogram
Histogram for position, velocity, acceleration, and jerk. <br>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
pos_diff = (dome.positionActual - dome.pos_cumtrap + 180) % 360 - 180
ax.hist(pos_diff, bins=20,color="darkorange")
ymin, ymax = ax.get_ylim()
ax.set_xlabel("Actual Position - Calculated Position(deg)")
ax.set_ylabel("N")
plt.yscale('log')
ax.autoscale(enable=True, axis='y', tight=True)
fig.savefig(plot_path /f"{dayobs}_position_histogram.png", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.hist([dome.velocityActual, dome.vel_sp], bins=20,label=["Actual Velocity","Calucated Velocity"],color=["grey","darkorange"])
ymin, ymax = ax.get_ylim()
ax.vlines(vel_max_limit,ymin=ymin, ymax=ymax,color='red', linestyle=':',label="Max Vel Limit")
ax.vlines(vel_min_limit,ymin=ymin, ymax=ymax,color='darkgreen', linestyle='-',label="Min Vel Limit")
ax.vlines(dome_setup.vmax.iloc[-1],ymin=ymin, ymax=ymax,color='Blue', linestyle=':',label="Applied Vel Limit")

ax.set_xlabel("Velocity (deg/s)")
ax.set_ylabel("N")
plt.yscale('log')
lgd = fig.legend(bbox_to_anchor=(1.4, 0.4),fontsize=12)
ax.autoscale(enable=True, axis='y', tight=True)
fig.savefig(plot_path /f"{dayobs}_vel_histogram.png",bbox_extra_artists=(lgd,), bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.hist([dome.vel_sp-dome.velocityActual], bins=30, label=r"${\Delta}$velocity")
ymin, ymax = ax.get_ylim()
plt.yscale('log')
ax.set_xlabel(r"${\Delta}$Velocity (deg/s)")
ax.set_ylabel("N")
fig.savefig(plot_path /f"{dayobs}_del_vel_histogram.png")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.hist([dome.accel], bins=20,label=["Actual Acceleration"],color=["darkorange"])
ax.set_xlim([-4,4])
ymin, ymax = ax.get_ylim()
ax.vlines(accel_max_limit,ymin=ymin, ymax=ymax,color='red', linestyle=':',label='Max Vel Limit')
ax.vlines(dome_setup.amax.iloc[-1],ymin=ymin, ymax=ymax,color='blue', linestyle=':')
ax.vlines(-1*accel_max_limit,ymin=ymin, ymax=ymax,color='red', linestyle=':')
ax.vlines(-1*dome_setup.amax.iloc[-1],ymin=ymin, ymax=ymax,color='blue', linestyle=':',label="Applied Max Vel Limit")
ax.vlines(accel_min_limit,ymin=ymin, ymax=ymax,color='green', linestyle=':',label="Max Vel Limit")
ax.vlines(-1*accel_min_limit,ymin=ymin, ymax=ymax,color='green', linestyle=':')

#ax.text(accel_max_limit*0.95, 0.65*ymax+0.35*ymin,'Max Vel Limit', horizontalalignment='center',        fontsize=12, color="red",rotation=90, verticalalignment='bottom')

#ax.text(accel_min_limit*0.95, 0.65*ymax+0.35*ymin,'Min Vel Limit', horizontalalignment='center',    fontsize=12, color="green",rotation=90, verticalalignment='bottom')

#ax.text(dome_setup.amax.iloc[-1]*1.1, 0.9*ymin+0.1*ymax,'Applied Max Vel Limit', horizontalalignment='center',  fontsize=12, color="Blue",rotation=90, verticalalignment='bottom')

plt.yscale('log')
lgd = fig.legend(bbox_to_anchor=(1.5, 0.4),fontsize=12)

ax.set_xlabel("Acceleration (deg/s$^{2}$)")
ax.set_ylabel("N")
ax.autoscale(enable=True, axis='y', tight=True)
fig.savefig(plot_path /f"{dayobs}_MTDome_accel_histogram.png",bbox_extra_artists=(lgd,), bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.hist([dome.jerk], bins=20,label=["Actual Jerk"],color=["darkorange"])
ax.set_xlim([-25,25])

ymin, ymax = ax.get_ylim()
ax.vlines(jerk_max_limit,ymin=ymin, ymax=ymax,color='red', linestyle=':',label="Max Jerk Limit")
ax.vlines(dome_setup.jmax.iloc[-1],ymin=ymin, ymax=ymax,color='blue', linestyle=':')
ax.vlines(-1*jerk_max_limit,ymin=ymin, ymax=ymax,color='red', linestyle=':')
ax.vlines(-1*dome_setup.jmax.iloc[-1],ymin=ymin, ymax=ymax,color='blue', linestyle=':',label="Applied Jerk Limit")
ax.vlines(jerk_min_limit,ymin=ymin, ymax=ymax,color='green', linestyle=':',label="Min Jerk Limit")
ax.vlines(-1*jerk_min_limit,ymin=ymin, ymax=ymax,color='green', linestyle=':')
ax.set_xlabel("Jerk (deg/s$^{3}$)")
ax.set_ylabel("N")
ax.autoscale(enable=True, axis='y', tight=True)
plt.yscale('log')
lgd = fig.legend(bbox_to_anchor=(1.4, 0.4),fontsize=12)

fig.savefig(plot_path /f"{dayobs}_MTDome_jerk_histogram.png",bbox_extra_artists=(lgd,), bbox_inches='tight')
